In [1]:
import pandas as pd
import numpy as np


In [9]:
df = pd.read_csv(r'application_train_all_models_final.csv', index_col=0)
print('Shape:', df.shape)
print('Колонки:', df.columns.tolist())

Shape: (307505, 92)
Колонки: ['SK_ID_CURR', 'TARGET', 'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'REGION_POPULATION_RELATIVE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_PHONE', 'FLAG_EMAIL', 'OCCUPATION_TYPE', 'CNT_FAM_MEMBERS', 'REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY', 'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START', 'REG_REGION_NOT_LIVE_REGION', 'REG_REGION_NOT_WORK_REGION', 'LIVE_REGION_NOT_WORK_REGION', 'REG_CITY_NOT_LIVE_CITY', 'REG_CITY_NOT_WORK_CITY', 'LIVE_CITY_NOT_WORK_CITY', 'ORGANIZATION_TYPE', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'OBS_30_CNT_SOCIAL_CIRCLE', 'DEF_30_CNT_SOCIAL_CIRCLE', 'OBS_60_CNT_SOCIAL_CIRCLE', 'DEF_60_CNT_SOCIAL_CIRCLE', 'DAYS_LAST_PHONE_C

## 1. Известный аномальный выброс — DAYS_EMPLOYED

Значение `365243` в `DAYS_EMPLOYED` — это технический placeholder для пенсионеров/безработных, не реальный стаж. Заменяем на NaN.

In [10]:
print('DAYS_EMPLOYED == 365243:', (df['DAYS_EMPLOYED'] == 365243).sum(), 'строк')
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
print('После замены NaN в DAYS_EMPLOYED:', df['DAYS_EMPLOYED'].isna().sum())

DAYS_EMPLOYED == 365243: 55374 строк
После замены NaN в DAYS_EMPLOYED: 55374


## 2. Клипинг по 99.9% перцентилю

Применяем к колонкам с потенциальными правыми выбросами. Бинарные флаги, нормализованные фичи (EXT_SOURCE, REGION_POPULATION_RELATIVE) и DAYS-колонки с отрицательными значениями — не трогаем.

In [11]:
# Колонки для клипинга — только те где возможны правые выбросы
clip_cols = [
    'AMT_INCOME_TOTAL',
    'AMT_CREDIT',
    'AMT_ANNUITY',
    'OBS_30_CNT_SOCIAL_CIRCLE',
    'DEF_30_CNT_SOCIAL_CIRCLE',
    'DEF_60_CNT_SOCIAL_CIRCLE',
    'AMT_REQ_CREDIT_BUREAU_HOUR',
    'AMT_REQ_CREDIT_BUREAU_DAY',
    'AMT_REQ_CREDIT_BUREAU_WEEK',
    'AMT_REQ_CREDIT_BUREAU_MON',
    'AMT_REQ_CREDIT_BUREAU_QRT',
    'AMT_REQ_CREDIT_BUREAU_YEAR',
]

# Оставляем только те колонки которые есть в датасете
clip_cols = [c for c in clip_cols if c in df.columns]

print(f'Клипинг {len(clip_cols)} колонок на уровне 99.9%:')
for col in clip_cols:
    before_max = df[col].max()
    cap = df[col].quantile(0.999)
    df[col] = df[col].clip(upper=cap)
    print(f'  {col}: max было {before_max:.1f} → теперь {cap:.1f}')

Клипинг 12 колонок на уровне 99.9%:
  AMT_INCOME_TOTAL: max было 117000000.0 → теперь 900000.0
  AMT_CREDIT: max было 4050000.0 → теперь 2517300.0
  AMT_ANNUITY: max было 258025.5 → теперь 110047.5
  OBS_30_CNT_SOCIAL_CIRCLE: max было 348.0 → теперь 17.0
  DEF_30_CNT_SOCIAL_CIRCLE: max было 34.0 → теперь 4.0
  DEF_60_CNT_SOCIAL_CIRCLE: max было 24.0 → теперь 3.0
  AMT_REQ_CREDIT_BUREAU_HOUR: max было 4.0 → теперь 1.0
  AMT_REQ_CREDIT_BUREAU_DAY: max было 9.0 → теперь 1.0
  AMT_REQ_CREDIT_BUREAU_WEEK: max было 8.0 → теперь 2.0
  AMT_REQ_CREDIT_BUREAU_MON: max было 27.0 → теперь 11.0
  AMT_REQ_CREDIT_BUREAU_QRT: max было 261.0 → теперь 4.0
  AMT_REQ_CREDIT_BUREAU_YEAR: max было 25.0 → теперь 9.0


## 3. Клипинг bureau-фич (если уже заджойнены)

In [12]:
bureau_clip_cols = [
    'bureau_amt_credit_sum',
    'bureau_amt_credit_mean',
    'bureau_amt_credit_max',
    'bureau_amt_debt_sum',
    'bureau_amt_debt_mean',
    'bureau_amt_overdue_max',
    'bureau_annuity_sum',
    'bureau_annuity_mean',
    'bureau_overdue_sum',
    'bureau_prolong_sum',
]

bureau_clip_cols = [c for c in bureau_clip_cols if c in df.columns]

if bureau_clip_cols:
    print(f'Клипинг {len(bureau_clip_cols)} bureau-фич на уровне 99.9%:')
    for col in bureau_clip_cols:
        before_max = df[col].max()
        cap = df[col].quantile(0.999)
        df[col] = df[col].clip(upper=cap)
        print(f'  {col}: max было {before_max:.1f} → теперь {cap:.1f}')
else:
    print('Bureau-фичи не найдены в датасете (добавь их через джойн сначала)')

Клипинг 10 bureau-фич на уровне 99.9%:
  bureau_amt_credit_sum: max было 1017957917.4 → теперь 30343248.4
  bureau_amt_credit_mean: max было 198072344.2 → теперь 8274081.0
  bureau_amt_credit_max: max было 396000000.0 → теперь 17550000.0
  bureau_amt_debt_sum: max было 334498331.2 → теперь 15214662.0
  bureau_amt_debt_mean: max было 43650000.0 → теперь 6150705.8
  bureau_amt_overdue_max: max было 115987185.0 → теперь 358948.3
  bureau_annuity_sum: max было 68504272.9 → теперь 1283459.8
  bureau_annuity_mean: max было 27282428.8 → теперь 239096.7
  bureau_overdue_sum: max было 5250.0 → теперь 1739.5
  bureau_prolong_sum: max было 9.0 → теперь 3.0


## 4. Итог

In [16]:
print('Shape после обработки выбросов:', df.shape)
print('\nПропуски после обработки:')
missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print(missing[missing > 0])

df.to_csv(r'application_train_all_models_final.csv')
print('\nСохранено!')

Shape после обработки выбросов: (307505, 92)

Пропуски после обработки:
Series([], dtype: float64)

Сохранено!


In [15]:
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].fillna(df['DAYS_EMPLOYED'].median())